# EODHD Bollinger — Colab
Upload the MK_BB_EODHD_Colab_Netlify.zip source package first. Add EODHD_API_TOKEN to Colab Secrets. This notebook does not publish to Netlify by itself.

In [ ]:
from google.colab import files, userdata
import os, zipfile
from pathlib import Path
uploaded = files.upload()
archive = next(name for name in uploaded if name.endswith('.zip'))
root = Path('/content/mk_bb_eodhd')
root.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    for member in z.infolist():
        dest = (root/member.filename).resolve()
        if not dest.is_relative_to(root.resolve()):
            raise ValueError('Unsafe archive path')
        if member.filename in ('config.json','universe.json') and dest.exists():
            print('Preserved:', member.filename)
            continue
        z.extract(member, root)
os.chdir(root)
os.environ['EODHD_API_TOKEN'] = userdata.get('EODHD_API_TOKEN')
%pip -q install -r requirements.txt


In [ ]:
!python -m unittest -v test_engine
!python bb_eodhd.py --init
!python bb_eodhd.py --catalog


## Symbol validation — v2.1
private/catalog.json contains real INDX and FOREX records. COMM is never queried. Check universe.json settings for unresolved indices. USD spot-metal pairs are verified when present in the catalog. Commodity date/value series are plotted at their native frequency via commodities.py; OHLCV cannot be produced, so stopped backtests stay disabled. Soybeans/Cocoa appear as unsupported slots.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'bb_eodhd.py'], check=True)
files.download('netlify_site.zip')


For daily automation, upload the .github/workflows/daily.yml file from the source package to a private GitHub repository. Define EODHD_API_TOKEN, NETLIFY_AUTH_TOKEN and NETLIFY_SITE_ID as Actions Secrets. After the first successful build and the access/license check, set the PUBLISH_APPROVED repository variable to true. See README.md.